# data cleaning and filtering

## data source

The data reused in this project is published on hugging face: https://huggingface.co/datasets/RyokoAI/ShareGPT52K. The original data was collected using the ShareGPT API is licensed with CC0: No Rights Reserved.

For more information check also the dataset card for information about the dataset summary and structure.

## data cleaning and filtering

The source dataset consists of two parts which are expected to primarily consist of messages in English and other Western languages. These datasets must be merged, and the English prompts must then be selected.

An initial attempt using the ‘langdetect’ library resulted in a dataset that contained Dutch and German messages as well as English ones. Furthermore, there were several compatibility issues with Python 3.13 and various libraries (e.g. fasttext also requires a C++ compiler).

Since the unit of analysis is a full conversation tree rather than a single short utterance, stopword-based language filtering is appropriate as a lightweight and transparent heuristic. Conversation trees contain enough natural-language context for function-word distributions to be informative. To reduce false positives, the filter compares English stopword evidence against other languages stopword evidence and applies a minimum margin, so ambiguous or mixed-language trees are excluded rather than incorrectly retained.

During random checks of selected data, other languages (e.g. Vietnamese, Korean) were identified, and corresponding stop words or characters were gradually incorporated into the code to filter these out. In the process, other major languages such as Spanish, Italian, etc. were also removed from the dataset.


Heuristics:
+ Python 3.13-compatible
+ No installation required
+ Easy to control
+ Fast enough
- Less robust in general

Library:
+ Better language detection overall
+ Detects more edge cases
- Installation issues
- Less transparent
- Sometimes overkill


Trade off/ Decision:
While a library is better for maintability, hard coded stopwords are better for portability: 

+ No installation required
+ Works reliably with Python 3.13
+ No external downloads
+ Fully reproducible
+ You can add specific DE/NL, other language markers


In the process of data preparation the decision was made to remove prompts with less than 5 words.


In [1]:
import json
import re
import random
from pathlib import Path
from time import perf_counter
import tiktoken
import pandas as pd
from spellchecker import SpellChecker
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

In [ ]:
WORD_RE = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ']+")

STOPWORDS = {
    "en": {
        "the", "and", "to", "of", "a", "in", "that", "is", "it", "for", "on",
        "with", "as", "this", "be", "are", "you", "your", "can", "will", "not",
        "or", "if", "we", "from", "by", "at", "an", "have", "has", "do", "does",
        "what", "which", "when", "where", "how", "why", "about", "would", "could",
        "should", "there", "their", "they", "them", "these", "those", "one",
        "more", "use", "using", "make", "write", "explain", "please", "help",
    },
    "de": {
        "der", "die", "das", "und", "ist", "ich", "du", "sie", "er", "es", "wir",
        "nicht", "ein", "eine", "einen", "zu", "mit", "auf", "für", "von", "wie",
        "was", "warum", "wenn", "dass", "kann", "sind", "haben", "werden",
        "auch", "oder", "aber", "bitte", "über", "hallo", "gebe", "spitznamen",
    },
    "nl": {
        "de", "het", "een", "en", "van", "ik", "je", "jij", "niet", "dat", "dit",
        "die", "op", "voor", "met", "als", "zijn", "hebben", "worden", "wat",
        "waar", "waarom", "hoe", "maar", "ook", "geen", "omdat", "maak", "eigen",
        "implementatie",
    },
    "vi": {
        "và", "là", "của", "có", "cho", "trong", "một", "không", "tôi", "bạn",
        "hãy", "này", "để", "với", "các", "những", "tính", "năng", "mô", "tả",
        "lợi", "ích", "người", "dùng", "sao", "theo", "trên", "sau",
    },
    "es": {
        "el", "la", "los", "las", "y", "de", "que", "en", "un", "una", "es",
        "por", "para", "con", "como", "no", "se", "del", "al", "qué", "cómo",
        "puede", "pueden", "usuario", "hacer", "contrato", "legislacion",
        "finanzas",
    },
    "fr": {
        "le", "la", "les", "et", "de", "des", "du", "un", "une", "est", "dans",
        "pour", "avec", "que", "qui", "pas", "sur", "ce", "cette", "comment",
        "vous", "nous", "peut", "faire",
    },
    "pt": {
        "o", "a", "os", "as", "e", "de", "que", "em", "um", "uma", "é", "para",
        "com", "como", "não", "por", "se", "do", "da", "dos", "das", "pode",
        "usuário", "fazer",
    },
    "it": {
        "il", "lo", "la", "gli", "le", "e", "di", "che", "in", "un", "una",
        "è", "per", "con", "come", "non", "si", "del", "della", "può", "fare",
        "utente",
    },
    "id": {
        "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "ini", "itu",
        "adalah", "saya", "kamu", "anda", "tidak", "bisa", "akan", "dalam",
        "pada", "sebagai", "atau", "karena", "apa", "bagaimana", "mengapa",
        "buat", "tulis", "jelaskan", "latar", "belakang", "alat", "mencuci",
        "tangan", "berbasis", "perkembangan", "terakhir", "bidang", "cara",
        "bikin",
    },
    "da": {
        "og", "i", "jeg", "det", "at", "en", "den", "til", "er", "som",
        "på", "de", "med", "han", "af", "for", "ikke", "der", "var",
        "mig", "sig", "men", "et", "har", "om", "vi", "min", "hvad",
        "betyder", "hvordan", "hvorfor", "kan", "du", "skrive", "lav",
    },
    "hi_romanized": {
        "yeh", "kya", "hy", "hai", "ka", "ki", "ke", "ma", "mein",
        "kaise", "kaisey", "batao", "banao", "likho", "suno", "yar",
        "dost", "shahi", "paneer", "banate", "hain",
    },

    "jp": {
        "的", "了", "在", "是", "我", "你", "请", "为", "和", "与"},

    "cn":{
        "の", "に", "を", "は", "が", "で", "と"},
    
    "kr":{
        "이", "그", "저", "것", "수", "있", "하"}
}

NON_ENGLISH_MARKERS = {
    "vi": set(
        "ăâđêôơư"
        "áàảãạ"
        "ắằẳẵặ"
        "ấầẩẫậ"
        "éèẻẽẹ"
        "ếềểễệ"
        "íìỉĩị"
        "óòỏõọ"
        "ốồổỗộ"
        "ớờởỡợ"
        "úùủũụ"
        "ứừửữự"
        "ýỳỷỹỵ"
    ),
    "de": set("äöüß"),
}


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)


def get_first_user_prompt(tree):
    conversations = tree.get("conversations", [])

    for message in conversations:
        if message.get("from") == "human":
            return message.get("value", "")

    return ""


def iter_text_values(node):
    if isinstance(node, str):
        yield node

    elif isinstance(node, list):
        for item in node:
            yield from iter_text_values(item)

    elif isinstance(node, dict):
        for key, value in node.items():
            if key.lower() in {"id", "parent", "parent_id", "children", "conversation_id"}:
                continue

            yield from iter_text_values(value)


def normalize_words(text):
    return [
        word.strip("'").lower()
        for word in WORD_RE.findall(text)
        if word.strip("'")
    ]


def count_words(text):
    return len(re.findall(r"[A-Za-z]+", text))


def non_latin_char_ratio(text):
    letters = [char for char in text if char.isalpha()]

    if not letters:
        return 0.0

    latin_letters = [
        char
        for char in letters
        if (
            "A" <= char <= "Z"
            or "a" <= char <= "z"
            or "À" <= char <= "Ö"
            or "Ø" <= char <= "ö"
            or "ø" <= char <= "ÿ"
        )
    ]

    return 1 - (len(latin_letters) / len(letters))


def has_non_english_marker(text):
    lowered = text.lower()

    for chars in NON_ENGLISH_MARKERS.values():
        if any(char in lowered for char in chars):
            return True

    return False


def count_stopwords(words):
    return {
        lang: sum(word in stopwords for word in words)
        for lang, stopwords in STOPWORDS.items()
    }


def get_language_scores(tree):
    text = "\n".join(iter_text_values(tree))
    words = [word for word in normalize_words(text) if len(word) > 1]

    return {
        "text": text,
        "words": words,
        "word_count": len(words),
        "stopword_scores": count_stopwords(words),
        "non_latin_ratio": non_latin_char_ratio(text),
        "has_non_english_marker": has_non_english_marker(text),
    }


def is_english_tree(
    tree,
    min_words=12,
    min_english_stopwords=3,
    min_margin=2,
    max_non_latin_ratio=0.05,
    min_first_prompt_words=5,
):
    scores = get_language_scores(tree)
    first_prompt = get_first_user_prompt(tree)

    if count_words(first_prompt) < min_first_prompt_words:
        return False

    if scores["word_count"] < min_words:
        return False

    if scores["non_latin_ratio"] > max_non_latin_ratio:
        return False

    if scores["has_non_english_marker"]:
        return False

    stopword_scores = scores["stopword_scores"]
    english_score = stopword_scores["en"]

    strongest_other_language = max(
        score
        for lang, score in stopword_scores.items()
        if lang != "en"
    )

    return (
        english_score >= min_english_stopwords
        and english_score - strongest_other_language >= min_margin
    )


def filter_english_trees(data):
    english_trees = []

    for tree in data:
        if is_english_tree(tree):
            english_trees.append(tree)

    return english_trees


def show_random_questions(data, n=10, max_chars=700):
    for index in random.sample(range(len(data)), min(n, len(data))):
        tree = data[index]

        texts = [
            text.strip()
            for text in iter_text_values(tree)
            if isinstance(text, str) and len(text.strip()) > 20
        ]

        if not texts:
            continue

        print("=" * 100)
        print(f"Index: {index}")
        print(texts[0][:max_chars])


part1 = load_json(
    "C:/Users/heike/Desktop/Stackfuel/Portfolio/llm-sustainability-analysis/01_data/01_raw/messy_dataset/sg_90k_part1.json"
)

part2 = load_json(
    "C:/Users/heike/Desktop/Stackfuel/Portfolio/llm-sustainability-analysis/01_data/01_raw/messy_dataset/sg_90k_part2.json"
)

data = part1 + part2

start = perf_counter()

english_data = filter_english_trees(data)

duration = perf_counter() - start

save_json(
    english_data,
    "C:/Users/heike/Desktop/Stackfuel/Portfolio/llm-sustainability-analysis/01_data/02_processed/sharegpt_english.json"
)

print(f"Original: {len(data)}")
print(f"English only: {len(english_data)}")
print(f"Removed: {len(data) - len(english_data)}")
print(f"Filtering and cleaning took: {duration:.2f} seconds")
print(f"Trees per second: {len(data) / duration:.0f}")

show_random_questions(english_data, n=10)


Original: 90665
English only: 53824
Removed: 36841
Filtering and cleaning took: 767.81 seconds
Trees per second: 118
Index: 3483
<div class="markdown prose w-full break-words dark:prose-invert light"><p>DALL-E 2 is a large language model developed by OpenAI. It can be used to generate text, images, and other forms of media based on natural language prompts. To use DALL-E 2, you will need to make an API request to the OpenAI API with your prompt, and the API will return the generated output. The API requires an API key, which can be obtained by creating an</p></div>
Index: 33617
The MS Internet Explorer vulnerability and why Chrome and Firefox aren't affected?
Index: 10953
Authenticating Guest Users for Firebase using  laravel
Index: 27983
Lets chat about ROS India Summit
Index: 2059
1 / 11.

Hello :) Today we are gonna create Images with a Diffusion model. I am gonna feed you some information about it.

I'm going to give you a verbatim copy of the manual that describes Midjourney and h